# Vi-DFD - Dataset Splitting → LMDB

Notebook chạy tuần tự từ trên xuống, dùng cấu trúc thư mục:

- `D:/BTL_DL/dataset/`: ảnh VinText gốc, **không chỉnh sửa**.
- `D:/BTL_DL/processed/vintext_notebook/`: `metadata.csv` và `crops/` đã có từ Task 1.
- `D:/BTL_DL/processed/vintext_notebook/splits/`: manifest đóng băng và báo cáo.
- `D:/BTL_DL/processed/vintext_notebook/lmdb/`: 4 LMDB train/val/test/unseen_test.

Các bước thực hiện: 
1. Kiểm tra dữ liệu đầu vào: Đọc metadata.csv, kiểm tra word crops và thống kê số lượng mẫu theo từng split.
2. Phát hiện ảnh trùng/gần trùng: Sử dụng SHA-256 và pHash để phát hiện, xác nhận và nhóm các ảnh scene trùng lặp.
3. Chia Train/Validation: Chia tập train gốc theo tỷ lệ 90/10 dựa trên ảnh gốc và nhóm ảnh trùng, tránh data leakage.
4. Giữ nguyên Test/Unseen Test: Bảo toàn hai tập đánh giá chính thức và ghi nhận các trường hợp trùng lặp giữa các split.
5. Đóng băng Split và Subsets: Lưu các split cố định và tạo tập con Diacritic-heavy, Rare combinations, Small crops để đánh giá chuyên sâu.
6. Xuất dữ liệu sang LMDB: Đóng gói word crops và transcript thành 4 LMDB train/val/test/unseen_test, sau đó kiểm tra khả năng đọc.


Notebook không chia ngẫu nhiên các word crops mà chia **theo ảnh scene gốc và các nhóm trùng đã xác nhận**. Tần suất tổ hợp dấu hiếm được tính **chỉ trên train**.

## Chuẩn bị thư viện và đường dẫn

In [1]:
# Chạy một lần nếu môi trường chưa có thư viện (bỏ dấu # để cài):
# %pip install pandas numpy opencv-python lmdb

import csv
import hashlib
import json
import random
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import lmdb
import numpy as np
import pandas as pd

DATASET_ROOT = Path(r"D:\BTL_DL\dataset")
PROCESSED_ROOT = Path(r"D:\BTL_DL\processed\vintext_notebook")
METADATA_PATH = PROCESSED_ROOT / "metadata.csv"
CROPS_ROOT = PROCESSED_ROOT / "crops"
SPLITS_ROOT = PROCESSED_ROOT / "splits"
LMDB_ROOT = PROCESSED_ROOT / "lmdb"
DUPLICATES_DIR = SPLITS_ROOT / "duplicates"
MANIFESTS_DIR = SPLITS_ROOT / "manifests"
SUBSETS_DIR = SPLITS_ROOT / "subsets"

SEED = 42
VAL_RATIO = 0.10
PHASH_AUTO_THRESHOLD = 4    # Tự động nhóm nếu pHash distance <= 4
PHASH_REPORT_THRESHOLD = 8  # Chỉ báo cáo nếu 5 <= distance <= 8
RARE_THRESHOLD = 20  # ít hơn 20 lần xuất hiện trên train
SMALL_CROP_HEIGHT = 16  # pixel trước khi thêm margin

SOURCE_FOLDERS = {
    "train": "train_images",
    "test": "test_image",
    "unseen_test": "unseen_test_images",
}

assert DATASET_ROOT.is_dir(), f"Không thấy dataset: {DATASET_ROOT}"
assert METADATA_PATH.is_file(), f"Chưa có Task 1 metadata: {METADATA_PATH}"
assert CROPS_ROOT.is_dir(), f"Chưa có crops: {CROPS_ROOT}"
for directory in (SPLITS_ROOT, DUPLICATES_DIR, MANIFESTS_DIR, SUBSETS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET_ROOT)
print("Metadata:", METADATA_PATH)
print("Duplicates:", DUPLICATES_DIR)
print("Manifests:", MANIFESTS_DIR)
print("Subsets:", SUBSETS_DIR)

Dataset: D:\BTL_DL\dataset
Metadata: D:\BTL_DL\processed\vintext_notebook\metadata.csv
Duplicates: D:\BTL_DL\processed\vintext_notebook\splits\duplicates
Manifests: D:\BTL_DL\processed\vintext_notebook\splits\manifests
Subsets: D:\BTL_DL\processed\vintext_notebook\splits\subsets


## 1. Kiểm tra metadata sau khi crop ảnh

File `metadata.csv` cần các cột `split`, `image_id`, `crop_path`, `transcript`, `original_word_height`. `crop_path` là đường dẫn tương đối tính từ `PROCESSED_ROOT`.

In [2]:
df = pd.read_csv(METADATA_PATH, encoding="utf-8-sig", keep_default_na=False)
required = {"split", "image_id", "crop_path", "transcript", "original_word_height"}
missing = required - set(df.columns)
assert not missing, f"Metadata thiếu các cột: {sorted(missing)}"
assert not df.empty, "metadata.csv rỗng"
assert set(df["split"]).issubset(SOURCE_FOLDERS), "Có split không xác định"

df["image_id"] = pd.to_numeric(df["image_id"], errors="raise").astype(int)
df["original_word_height"] = pd.to_numeric(df["original_word_height"], errors="raise")
df["transcript"] = df["transcript"].astype(str).map(lambda s: unicodedata.normalize("NFC", s))
assert df["transcript"].str.len().gt(0).all(), "Có transcript rỗng"
assert not df["transcript"].eq("###").any(), "Metadata vẫn có ###"
assert df["original_word_height"].ge(0).all(), "Chiều cao crop không hợp lệ"

missing_crops = df.loc[~df["crop_path"].map(lambda p: (PROCESSED_ROOT / str(p)).is_file())]
assert missing_crops.empty, f"Thiếu {len(missing_crops)} file crop, ví dụ: {missing_crops['crop_path'].head().tolist()}"

print("Tổng word crops:", len(df))
display(df.groupby("split", sort=False).agg(word_crops=("image_id", "size"), source_images=("image_id", "nunique")))
display(df.head(3))

Tổng word crops: 43045


,word_crops,source_images
split,,
train,25776,1199
test,7201,297
unseen_test,10068,499


,split,image_id,source_image,annotation_file,line_number,crop_path,transcript,crop_width,crop_height,original_word_height,original_word_width
0,train,1,train_images\im0001.jpg,gt_1.txt,2,crops/train/im0001_line0002.png,CHẤT,24,39,32,24
1,train,1,train_images\im0001.jpg,gt_1.txt,3,crops/train/im0001_line0003.png,LƯỢNG,30,39,32,30
2,train,1,train_images\im0001.jpg,gt_1.txt,4,crops/train/im0001_line0004.png,TỐT,16,42,35,16


## 2. Tạo chỉ mục ảnh gốc và hash

SHA-256 phát hiện file trùng byte; pHash 64-bit cho phép phát hiện ảnh có nội dung thị giác tương tự. **Không dùng hash của word crop để chia train/val.**

In [3]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def phash_file(path: Path) -> int:
    # cv2.imdecode tương thích đường dẫn Windows có Unicode.
    image = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Không đọc được ảnh: {path}")
    image = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(image)
    low = dct[:8, :8].flatten()
    med = float(np.median(low[1:]))  # bỏ hệ số DC khỏi median
    bits = low > med
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


images = []
for source_split, folder_name in SOURCE_FOLDERS.items():
    folder = DATASET_ROOT / folder_name
    assert folder.is_dir(), f"Thiếu thư mục: {folder}"
    for path in sorted(folder.glob("*.jpg")):
        assert path.stem.lower().startswith("im") and path.stem[2:].isdigit(), path
        image_id = int(path.stem[2:])
        images.append({
            "source_split": source_split,
            "image_id": image_id,
            "image_file": str(path.relative_to(DATASET_ROOT).as_posix()),
            "sha256": sha256_file(path),
            "phash": f"{phash_file(path):016x}",
        })
images_df = pd.DataFrame(images)
assert not images_df.empty, "Không tìm thấy ảnh gốc"
assert not images_df.duplicated("image_id").any(), "image_id lặp trong nhiều thư mục"

# Kiểm tra metadata thực sự tham chiếu đúng split gốc.
source_lookup = images_df.set_index("image_id")["source_split"].to_dict()
unknown_ids = set(df["image_id"]) - set(source_lookup)
assert not unknown_ids, f"Có image_id không tồn tại trong dataset: {list(unknown_ids)[:10]}"
assert (df["split"] == df["image_id"].map(source_lookup)).all(), "Metadata split không khớp ảnh gốc"

images_df.to_csv(DUPLICATES_DIR / "source_image_hashes.csv", index=False, encoding="utf-8-sig")
print("Ảnh gốc:", len(images_df))
display(images_df.groupby("source_split").size().rename("images").to_frame())

Ảnh gốc: 2000


,images
source_split,
test,300
train,1200
unseen_test,500


## 3. Tự động phân loại cặp ảnh trùng

- `duplicates/sha256_exact_pairs.csv`: ảnh trùng byte, tự động nhóm.
- `duplicates/phash_auto_pairs.csv`: các cặp có pHash distance **≤ 4**, tự động nhóm.
- `duplicates/phash_report_pairs.csv`: các cặp có pHash distance **5–8**, **chỉ để tham khảo**, không nhóm.


In [4]:
records = images_df.to_dict("records")
exact_pairs = []
auto_pairs = []
report_pairs = []

pair_columns = [
    "image_id_a", "image_id_b", "file_a", "file_b",
    "split_a", "split_b", "phash_distance",
]

# 2.000 ảnh ~2 triệu cặp: có thể xử lý trực tiếp.
for i, a in enumerate(records):
    ha = int(a["phash"], 16)
    for b in records[i + 1:]:
        same_bytes = a["sha256"] == b["sha256"]
        dist = (ha ^ int(b["phash"], 16)).bit_count()
        item = {
            "image_id_a": a["image_id"],
            "image_id_b": b["image_id"],
            "file_a": a["image_file"],
            "file_b": b["image_file"],
            "split_a": a["source_split"],
            "split_b": b["source_split"],
            "phash_distance": dist,
        }
        if same_bytes:
            exact_pairs.append(item)
        elif dist <= PHASH_AUTO_THRESHOLD:
            auto_pairs.append(item)
        elif dist <= PHASH_REPORT_THRESHOLD:
            report_pairs.append(item)

exact_df = pd.DataFrame(exact_pairs, columns=pair_columns)
auto_df = pd.DataFrame(auto_pairs, columns=pair_columns)
report_df = pd.DataFrame(report_pairs, columns=pair_columns)

exact_df.to_csv(DUPLICATES_DIR / "sha256_exact_pairs.csv", index=False, encoding="utf-8-sig")
auto_df.to_csv(DUPLICATES_DIR / "phash_auto_pairs.csv", index=False, encoding="utf-8-sig")
report_df.to_csv(DUPLICATES_DIR / "phash_report_pairs.csv", index=False, encoding="utf-8-sig")

# Bỏ qua file review cũ để không gây hiểu nhầm;
# không tự xóa nhằm giữ lại lịch sử quyết định nếu đã có.
old_review = SPLITS_ROOT / "phash_review.csv"
if old_review.exists():
    print("Lưu ý: file phash_review.csv cũ vẫn còn nhưng notebook KHÔNG đọc file này:", old_review)

print("SHA-256 exact duplicate pairs:", len(exact_df))
print(f"Auto-group pHash pairs (distance <= {PHASH_AUTO_THRESHOLD}):", len(auto_df))
print(f"Report-only pHash pairs ({PHASH_AUTO_THRESHOLD + 1}–{PHASH_REPORT_THRESHOLD}):", len(report_df))
display(auto_df.head(10))


SHA-256 exact duplicate pairs: 0
Auto-group pHash pairs (distance <= 4): 2
Report-only pHash pairs (5–8): 1


,image_id_a,image_id_b,file_a,file_b,split_a,split_b,phash_distance
0,478,593,train_images/im0478.jpg,train_images/im0593.jpg,train,train,4
1,894,1670,train_images/im0894.jpg,unseen_test_images/im1670.jpg,train,unseen_test,4


## 4. Nhóm trùng tự động và báo cáo trùng xuyên split chính thức

Nhóm bằng Union-Find từ các cặp SHA-256 trùng byte và pHash distance ≤ ngưỡng. Không tự ý gộp hoặc di chuyển ảnh giữa tập test/unseen_test và train.

In [5]:
class UnionFind:
    def __init__(self, items):
        self.parent = {int(x): int(x) for x in items}

    def find(self, x):
        x = int(x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)

uf = UnionFind(images_df["image_id"].tolist())
for row in exact_df.itertuples(index=False):
    uf.union(row.image_id_a, row.image_id_b)
for row in auto_df.itertuples(index=False):
    uf.union(row.image_id_a, row.image_id_b)

images_df["duplicate_group_id"] = images_df["image_id"].map(uf.find)
images_df.to_csv(DUPLICATES_DIR / "duplicate_groups.csv", index=False, encoding="utf-8-sig")

groups = images_df.groupby("duplicate_group_id").agg(
    images=("image_id", "size"),
    source_splits=("source_split", lambda s: ",".join(sorted(set(s))))
).reset_index()
cross = groups[groups["source_splits"].str.contains(",", regex=False)]
cross.to_csv(DUPLICATES_DIR / "cross_official_split_duplicates.csv", index=False, encoding="utf-8-sig")

print("Nhóm có >= 2 ảnh:", int((groups["images"] > 1).sum()))
print("Nhóm vượt ranh giới train/test/unseen:", len(cross))
display(cross.head(20))
if len(cross):
    print("CẢNH BÁO: Có nhóm trùng giữa split chính thức; giữ nguyên test/unseen_test.")
    print("Cần ghi nhận nguy cơ data leakage khi diễn giải kết quả đánh giá.")


Nhóm có >= 2 ảnh: 2
Nhóm vượt ranh giới train/test/unseen: 1


,duplicate_group_id,images,source_splits
892,894,2,"train,unseen_test"


CẢNH BÁO: Có nhóm trùng giữa split chính thức; giữ nguyên test/unseen_test.
Cần ghi nhận nguy cơ data leakage khi diễn giải kết quả đánh giá.


## 5. Chia train/val theo nhóm ảnh scene (seed cố định)

Chia khoảng **ảnh scene train gốc** vào val, giữ tất cả word crops của cùng ảnh và cùng nhóm trùng trong cùng split. Test và unseen_test không đổi.

In [6]:
train_images = images_df[images_df["source_split"].eq("train")].copy()
train_groups = [g["image_id"].tolist() for _, g in train_images.groupby("duplicate_group_id")]
rng = random.Random(SEED)
rng.shuffle(train_groups)

target = round(len(train_images) * VAL_RATIO)
selected_val = set()
# Greedy selection: cố gắng gần 10% số ảnh, không xé nhóm trùng.
for group_ids in train_groups:
    if len(selected_val) >= target:
        break
    if abs(target - (len(selected_val) + len(group_ids))) <= abs(target - len(selected_val)):
        selected_val.update(group_ids)
# Trường hợp không có nhóm nào được chọn (ví dụ tập rất nhỏ)
assert selected_val and len(selected_val) < len(train_images), "Không tạo được train/val hợp lệ"

image_split = {}
for row in images_df.itertuples(index=False):
    if row.source_split == "train":
        image_split[int(row.image_id)] = "val" if row.image_id in selected_val else "train"
    else:
        image_split[int(row.image_id)] = row.source_split

# Chứng minh không có cùng duplicate_group trong train và val.
train_images["final_split"] = train_images["image_id"].map(image_split)
assert train_images.groupby("duplicate_group_id")["final_split"].nunique().max() == 1

manifests = {}
for split in ["train", "val", "test", "unseen_test"]:
    subset = df[df["image_id"].map(image_split).eq(split)].copy()
    subset.insert(0, "final_split", split)
    # Giữ thứ tự tái lập được; không phải thứ tự shuffle trong quá trình train.
    sort_cols = ["image_id"] + (["line_number"] if "line_number" in subset.columns else ["crop_path"])
    subset = subset.sort_values(sort_cols, kind="stable").reset_index(drop=True)
    assert not subset.empty, f"Tập {split} rỗng"
    subset.to_csv(MANIFESTS_DIR / f"{split}.csv", index=False, encoding="utf-8-sig")
    manifests[split] = subset

image_assignments = images_df[["image_id", "source_split", "duplicate_group_id"]].copy()
image_assignments["final_split"] = image_assignments["image_id"].map(image_split)
image_assignments.to_csv(MANIFESTS_DIR / "image_split_assignments.csv", index=False, encoding="utf-8-sig")

print(f"Validation scene images: {len(selected_val)}/{len(train_images)} ({len(selected_val)/len(train_images):.1%})")
for split, sub in manifests.items():
    print(f"{split:<12}: {len(sub):>6} crops / {sub['image_id'].nunique():>4} scene images")
assert set(manifests["train"]["image_id"]).isdisjoint(manifests["val"]["image_id"])

Validation scene images: 120/1200 (10.0%)
train       :  23269 crops / 1079 scene images
val         :   2507 crops /  120 scene images
test        :   7201 crops /  297 scene images
unseen_test :  10068 crops /  499 scene images


## 6. Đóng băng tập con đánh giá

- **Diacritic-heavy**: word có ít nhất 2 ký tự mang dấu hoặc biến thể chữ có dấu (kiểm tra Unicode NFD).

-> Mục đích: Kiểm tra khả năng nhận dạng từ nhiều dấu.
- **Rare combinations**: tổ hợp ký tự cơ sở + dấu xuất hiện ít hơn 20 lần trong **train**, đánh giá trên `val/test/unseen_test`.

-> Mục đích: Kiểm tra khả năng nhận dạng tổ hợp dấu hiếm.
- **Small crops**: chiều cao vùng chữ gốc (trước margin) dưới 16 px.

-> Mục đích: Kiểm tra khả năng nhận dạng chữ nhỏ

Không dùng `test/unseen` để xác định từ vựng tổ hợp hiếm.

In [7]:
def marked_char_combinations(text: str):
    """Các tổ hợp (ký tự cơ sở, dấu kết hợp) sau NFD; bao gồm ă/â/ê/ô/ơ/ư/đ."""
    result = []
    for ch in unicodedata.normalize("NFC", text):
        if ch.isspace():
            continue
        # đ/Đ không được tách ra bằng NFD, xử lý riêng.
        if ch in "đĐ":
            result.append(("d" if ch == "đ" else "D", "stroke"))
            continue
        decomp = unicodedata.normalize("NFD", ch)
        if len(decomp) > 1 and any(unicodedata.combining(c) for c in decomp[1:]):
            base = decomp[0].lower()
            marks = "+".join(f"U+{ord(c):04X}" for c in decomp[1:])
            result.append((base, marks))
    return result


def combo_key(combo):
    return f"{combo[0]}|{combo[1]}"

train_counter = Counter(
    combo_key(comb)
    for text in manifests["train"]["transcript"]
    for comb in marked_char_combinations(str(text))
)
rare_keys = {k for k, count in train_counter.items() if count < RARE_THRESHOLD}

combo_counts = pd.DataFrame(
    [{"combination": k, "train_frequency": v, "rare": v < RARE_THRESHOLD}
     for k, v in sorted(train_counter.items())],
    columns=["combination", "train_frequency", "rare"]
)
combo_counts.to_csv(SUBSETS_DIR / "train_diacritic_combination_counts.csv", index=False, encoding="utf-8-sig")

subset_results = []
for split in ["val", "test", "unseen_test"]:
    sub = manifests[split].copy()
    combos_by_row = sub["transcript"].map(lambda t: [combo_key(c) for c in marked_char_combinations(str(t))])
    masks = {
        "diacritic_heavy": combos_by_row.map(lambda c: len(c) >= 2),
        "rare_combinations": combos_by_row.map(lambda c: any(train_counter.get(x, 0) < RARE_THRESHOLD for x in c)),
        "small_crops": sub["original_word_height"].lt(SMALL_CROP_HEIGHT),
    }
    for name, mask in masks.items():
        path = SUBSETS_DIR / f"{split}_{name}.csv"
        sub.loc[mask].to_csv(path, index=False, encoding="utf-8-sig")
        subset_results.append({"split": split, "subset": name, "word_crops": int(mask.sum()), "file": path.name})

display(pd.DataFrame(subset_results))
print("Số tổ hợp dấu train:", len(train_counter), "| tổ hợp hiếm:", len(rare_keys))

,split,subset,word_crops,file
0,val,diacritic_heavy,183,val_diacritic_heavy.csv
1,val,rare_combinations,6,val_rare_combinations.csv
2,val,small_crops,489,val_small_crops.csv
3,test,diacritic_heavy,587,test_diacritic_heavy.csv
4,test,rare_combinations,24,test_rare_combinations.csv
5,test,small_crops,1545,test_small_crops.csv
6,unseen_test,diacritic_heavy,776,unseen_test_diacritic_heavy.csv
7,unseen_test,rare_combinations,33,unseen_test_rare_combinations.csv
8,unseen_test,small_crops,1916,unseen_test_small_crops.csv


Số tổ hợp dấu train: 67 | tổ hợp hiếm: 5


## 7. Ghi Freeze Manifest và checksums

Lưu random seed, ngưỡng pHash tự động, số mẫu và checksum. `phash_report_pairs.csv` là báo cáo, không tham gia chia split.
Các CSV nằm trong `duplicates/`, `manifests/`, `subsets/`; checksum được ghi theo đường dẫn tương đối từ `splits/`.

- Freeze Manifest: một file JSON ghi lại dataset đã được tạo như thế nào.
- Checksums: một giá trị được tính từ nội dung file, dùng để kiểm tra file có thay đổi hay không.

-> Mục đích: Kiểm tra xem các mô hình có sử dụng đúng cùng phiên bản dữ liệu không.

In [8]:
def file_sha256(path):
    return sha256_file(path)

manifest_paths = sorted(
    p for folder in (DUPLICATES_DIR, MANIFESTS_DIR, SUBSETS_DIR)
    for p in folder.glob("*.csv")
)

freeze = {
    "seed": SEED,
    "validation_ratio_target": VAL_RATIO,
    "phash_auto_group_threshold": PHASH_AUTO_THRESHOLD,
    "phash_report_only_max_distance": PHASH_REPORT_THRESHOLD,
    "automatic_duplicate_grouping": True,
    "rare_frequency_threshold_train_only": RARE_THRESHOLD,
    "small_crop_original_height_lt": SMALL_CROP_HEIGHT,
    "word_crop_counts": {k: len(v) for k, v in manifests.items()},
    "image_counts": {k: int(v["image_id"].nunique()) for k, v in manifests.items()},
    "metadata_sha256": sha256_file(METADATA_PATH),
    "manifest_sha256": {p.relative_to(SPLITS_ROOT).as_posix(): file_sha256(p) for p in manifest_paths},
    "cross_official_split_duplicate_groups": int(len(cross)),
    "notes": [
        "SHA-256 identical files and pHash distance <= auto threshold are grouped automatically (heuristic)",
        "Official test and unseen_test image assignments are unchanged",
        "Rare combinations defined using training images only",
    ],
}
freeze_path = SPLITS_ROOT / "freeze_manifest.json"
freeze_path.write_text(json.dumps(freeze, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã đóng băng:", freeze_path)
print(json.dumps(freeze["word_crop_counts"], indent=2))

Đã đóng băng: D:\BTL_DL\processed\vintext_notebook\splits\freeze_manifest.json
{
  "train": 23269,
  "val": 2507,
  "test": 7201,
  "unseen_test": 10068
}


## 8. Xuất 4 LMDB độc lập

Format key kiểu STR thông dụng:

- `num-samples` → số mẫu
- `image-000000001` → bytes ảnh PNG
- `label-000000001` → transcript UTF-8 (NFC)
- `source-id-000000001` → `image_id:line_number` (để truy vết)

**Lưu ý:** Không tạo lại LMDB nếu thư mục đã có dữ liệu, tránh ghi đè không chủ ý. Xóa/move các thư mục LMDB cũ thủ công nếu muốn rebuild sau khi đổi split. DataLoader sau này phải dùng đúng key schema này.

In [9]:
def build_one_lmdb(split, manifest, out_dir, commit_every=500):
    if out_dir.exists() and any(out_dir.iterdir()):
        raise FileExistsError(
            f"LMDB đã tồn tại: {out_dir}. Để tránh ghi đè, hãy di chuyển/xóa thư mục cũ trước khi chạy lại."
        )
    out_dir.mkdir(parents=True, exist_ok=True)

    # map_size là sức chứa tối đa của LMDB, không phải dung lượng thật luôn được dùng.
    image_bytes_total = sum((PROCESSED_ROOT / str(p)).stat().st_size for p in manifest["crop_path"])
    map_size = max(1 << 28, int(image_bytes_total * 2.5) + (64 << 20))
    env = lmdb.open(str(out_dir), map_size=map_size, subdir=True, lock=True)
    try:
        txn = env.begin(write=True)
        for idx, row in enumerate(manifest.itertuples(index=False), start=1):
            path = PROCESSED_ROOT / str(row.crop_path)
            image_data = path.read_bytes()
            label = unicodedata.normalize("NFC", str(row.transcript))
            suffix = f"{idx:09d}".encode("ascii")
            line_number = getattr(row, "line_number", "")
            source_id = f"{int(row.image_id)}:{line_number}".encode("utf-8")
            txn.put(b"image-" + suffix, image_data)
            txn.put(b"label-" + suffix, label.encode("utf-8"))
            txn.put(b"source-id-" + suffix, source_id)
            if idx % commit_every == 0:
                txn.commit()
                txn = env.begin(write=True)
        txn.put(b"num-samples", str(len(manifest)).encode("ascii"))
        txn.commit()
        env.sync()
    finally:
        env.close()
    return len(manifest)


for split in ["train", "val", "test", "unseen_test"]:
    count = build_one_lmdb(split, manifests[split], LMDB_ROOT / split)
    print(f"{split:<12}: {count:>6} samples -> {LMDB_ROOT / split}")

train       :  23269 samples -> D:\BTL_DL\processed\vintext_notebook\lmdb\train
val         :   2507 samples -> D:\BTL_DL\processed\vintext_notebook\lmdb\val
test        :   7201 samples -> D:\BTL_DL\processed\vintext_notebook\lmdb\test
unseen_test :  10068 samples -> D:\BTL_DL\processed\vintext_notebook\lmdb\unseen_test


## 9. Đọc thử LMDB và kiểm tra kết quả

In [10]:
for split in ["train", "val", "test", "unseen_test"]:
    folder = LMDB_ROOT / split
    env = lmdb.open(str(folder), readonly=True, lock=False, readahead=False, max_readers=1)
    try:
        with env.begin() as txn:
            n_raw = txn.get(b"num-samples")
            assert n_raw is not None
            n = int(n_raw.decode("ascii"))
            assert n == len(manifests[split]), f"Số mẫu không khớp: {split}"
            assert n > 0
            image_data = txn.get(b"image-000000001")
            label_data = txn.get(b"label-000000001")
            assert image_data and label_data, f"Thiếu sample đầu tiên: {split}"
            decoded = cv2.imdecode(np.frombuffer(image_data, np.uint8), cv2.IMREAD_COLOR)
            assert decoded is not None and decoded.size > 0
            print(f"{split:<12}: OK | {n} samples | first label = {label_data.decode('utf-8')!r} | crop = {decoded.shape[1]}x{decoded.shape[0]}")
    finally:
        env.close()

print("\nHOÀN THÀNH TASK 3. Lưu và chia sẻ các CSV/JSON trong splits để tái lập thí nghiệm.")

train       : OK | 23269 samples | first label = 'CHẤT' | crop = 24x39
val         : OK | 2507 samples | first label = 'NHÀ' | crop = 157x204
test        : OK | 7201 samples | first label = 'CÀ' | crop = 28x42
unseen_test : OK | 10068 samples | first label = 'VIET' | crop = 64x29

HOÀN THÀNH TASK 3. Lưu và chia sẻ các CSV/JSON trong splits để tái lập thí nghiệm.


## Checklist sau khi chạy

- [ ] Đã xem thống kê `phash_auto_pairs.csv` và `cross_official_split_duplicates.csv` (không cần nhập `decision`).
- [ ] Đã kiểm tra số lượng train/val/test/unseen và các subset.
- [ ] Đã lưu manifest CSV và `freeze_manifest.json` để tái lập.
- [ ] Cả 4 LMDB được tạo và đọc thử thành công.

**Lưu ý:** Nếu đã tạo LMDB từ lần chạy trước, cần **di chuyển hoặc xóa riêng các thư mục LMDB cũ** trước khi chạy lại cell xuất LMDB. Không xóa ảnh gốc hoặc crops. Tự động nhóm bằng pHash có thể gây false positives/false negatives, nên báo cáo rõ ngưỡng và phương pháp.